# SQL Analysis: Lending Club Loans (2007–2011)

This notebook picks up where `01_eda.ipynb` left off. The dataset is loaded into a SQLite database, and the questions below are answered using SQL rather than pandas.

**Findings covered:**
1. Charge-off rate by loan purpose
2. Highest-interest-rate loan per purpose
3. Best average loan amount by grade & employment length
4. Interest rate trend over time (month-over-month)


## Setup: load data and clean known issues

Two columns need cleaning **before** they're written into SQLite, otherwise SQL queries silently produce wrong results:

- `int_rate` arrives as text with a `%` sign (e.g. `"10.00%"`) — left as-is, SQL sorts/compares it as a string, not a number (e.g. `"9.99%"` sorts as "greater than" `"10.00%"` alphabetically).
- `issue_d` arrives as text (e.g. `"Apr-08"`) — left as-is, SQL sorts it alphabetically rather than chronologically.

Both are fixed in pandas first, then written into the database.

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv("loan.csv")

# Clean int_rate: strip '%' and convert to float
df["int_rate"] = df["int_rate"].astype(str).str.rstrip("%").astype(float)

# Clean issue_d: parse into a real datetime (format is e.g. "Apr-08" -> month-year)
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%y")

conn = sqlite3.connect("lending_club.db")
df.to_sql("loans", conn, if_exists="replace", index=False)

## Sanity check — grade vs. interest rate

Quick check that the data loaded correctly: average interest rate should climb steadily from grade A (lowest risk) to grade G (highest risk).

In [ ]:
query = """
SELECT grade, COUNT(*) AS num_loans, AVG(int_rate) AS avg_rate
FROM loans
GROUP BY grade
ORDER BY grade
"""
pd.read_sql(query, conn)

## Finding 1 — Charge-off rate by loan purpose

**Question:** For each loan purpose, what proportion of loans were ultimately charged off (i.e. not repaid)?

**Method:** `CASE WHEN loan_status = 'Charged Off' THEN 1.0 ELSE 0 END` turns each row into a 0/1 flag; averaging that column gives the proportion. `HAVING COUNT(*) >= 200` excludes purposes with too few loans to give a reliable rate.

**Result:** `small_business` loans have the highest charge-off rate at roughly 27% — more than double several other categories. This lines up with intuition: small businesses are a fundamentally riskier lending category than, say, a wedding or a car loan.

In [ ]:
query = """
SELECT purpose,
       COUNT(*) AS num_loans,
       AVG(CASE WHEN loan_status = 'Charged Off' THEN 1.0 ELSE 0 END) AS charge_off_rate
FROM loans
GROUP BY purpose
HAVING COUNT(*) >= 200
ORDER BY charge_off_rate DESC
"""
pd.read_sql(query, conn)

## Finding 2 — Highest interest rate loan, per purpose

**Question:** For each loan purpose, which single loan carried the highest interest rate?

**Method:** A CTE (`ranked_dataset`) ranks loans within each `purpose` group by `int_rate` (highest first) using `RANK() OVER (PARTITION BY purpose ORDER BY int_rate DESC)`. The outer query then filters to rank 1 per group.

**Result:** Car loans surface among the purposes with the single highest recorded rate — somewhat counterintuitive, since auto loans are typically secured (backed by the vehicle as collateral) and are often priced lower than unsecured personal loans. This is worth flagging as a "single extreme outlier per group" rather than a trend — it reflects one loan, not the typical car loan rate (see the grade-level averages above for the more representative picture).

In [ ]:
query = """
WITH ranked_dataset AS (
    SELECT purpose, int_rate,
        RANK() OVER (PARTITION BY purpose ORDER BY int_rate DESC) AS ranking
    FROM loans
)
SELECT purpose, int_rate
FROM ranked_dataset
WHERE ranking = 1
ORDER BY purpose
"""
pd.read_sql(query, conn)

## Finding 3 — Best average loan amount by grade & employment length

**Question:** Within each loan grade, which employment-length group receives the largest average loan amount?

**Method:** A CTE (`by_grade_length_amnt`) first aggregates to one row per `grade` + `emp_length` combination using `GROUP BY`. The outer query then ranks those aggregated rows within each grade using `RANK() OVER (PARTITION BY grade ORDER BY avg_loan_amnt DESC)`, and filters to the top-ranked group per grade.

**Result:** Longer-tenured borrowers (8+ years, 10+ years) tend to receive the largest average loan amounts within their grade — consistent with the Phase 1 finding that higher income unlocks larger loans, since longer employment tenure often correlates with higher income.

In [ ]:
query = """
WITH by_grade_length_amnt AS (
    SELECT grade, emp_length,
        AVG(loan_amnt) AS avg_loan_amnt
    FROM loans
    GROUP BY grade, emp_length
),
ranked AS (
    SELECT grade, emp_length, avg_loan_amnt,
        RANK() OVER (PARTITION BY grade ORDER BY avg_loan_amnt DESC) AS ranking
    FROM by_grade_length_amnt
)
SELECT grade, emp_length, avg_loan_amnt
FROM ranked
WHERE ranking = 1
ORDER BY grade
"""
pd.read_sql(query, conn)

## Finding 4 — Interest rate trend over time (month-over-month)

**Question:** How did the average interest rate charged on new loans change from 2007 to 2011?

**Method:** A CTE (`monthly_avg`) aggregates to one average `int_rate` per `issue_d` month. The outer query then uses `LAG(avg_rate) OVER (ORDER BY issue_d)` to pull in the previous month's average, and computes the change.

**Data quality note:** the very first month, June 2007, contains only a single loan (`n=1`) — its average is really just one loan's rate, not a meaningful monthly average. Months with fewer than 10 loans are excluded via `HAVING COUNT(*) >= 10` to avoid this kind of small-sample noise.

**Result:** average interest rates drift upward over the four-year window — from roughly 9–11% in 2007–2008 to over 13% by the end of 2011 — with some fluctuation month to month rather than a perfectly smooth climb. This lines up with the Phase 1 finding that loan volume grew ~100x over the same period, suggesting Lending Club may have extended credit to a somewhat riskier borrower pool as it scaled.

In [ ]:
query = """
WITH monthly_avg AS (
    SELECT issue_d, AVG(int_rate) AS avg_rate, COUNT(*) AS num_loans
    FROM loans
    GROUP BY issue_d
    HAVING COUNT(*) >= 10
)
SELECT issue_d, avg_rate, num_loans,
    LAG(avg_rate) OVER (ORDER BY issue_d) AS prev_month_rate,
    avg_rate - LAG(avg_rate) OVER (ORDER BY issue_d) AS rate_change
FROM monthly_avg
ORDER BY issue_d
"""
pd.read_sql(query, conn)

### Visualizing the trend

In [ ]:
import matplotlib.pyplot as plt

trend = pd.read_sql("""
    SELECT issue_d, AVG(int_rate) AS avg_rate, COUNT(*) AS num_loans
    FROM loans
    GROUP BY issue_d
    HAVING COUNT(*) >= 10
    ORDER BY issue_d
""", conn, parse_dates=["issue_d"])

plt.figure(figsize=(10, 5))
plt.plot(trend["issue_d"], trend["avg_rate"])
plt.title("Average Interest Rate by Month (2007–2011)")
plt.xlabel("Issue Date")
plt.ylabel("Average Interest Rate (%)")
plt.tight_layout()
plt.show()

## Summary — so what?

This notebook doesn't uncover statistically "new" facts beyond the Phase 1 pandas EDA — the same dataset, after all, can only say what it says. What it does demonstrate is fluency with the SQL patterns that come up constantly in interviews and on the job:

- **Aggregate filtering** (`GROUP BY` + `HAVING` + `CASE WHEN`) to compute conditional rates
- **Window functions** (`RANK`, `LAG`) to rank within groups and compare rows to their neighbors
- **CTEs** to break multi-step logic into clear, named, reusable pieces
- **Data quality vigilance** — catching a single-loan month before it distorted a trend line, and catching text-vs-numeric type issues before they silently broke a sort

Next step: Phase 3 — statistics, starting with distributions, sampling, and the Central Limit Theorem.